# 09_noise_robust -- Production Ensemble (5-Seed Deep Ensemble)
### Langkah terakhir sebelum benar-benar difiksasi sebagai model produksi

**Konteks:** `CBQD - HPO Seed Stability.ipynb` menemukan `09_noise_robust` (tuned) punya
mean test macro-F1 0,9540 +/- 0,0171 across 5 seed (42-46) -- stabil, tapi tetap ada variance
run-to-run yang nyata. `CBQD - XAI 09-Noise-Robust Full.ipynb` memberi checkpoint tuned
(seed tunggal, test-F1=0,9739) bukti XAI paling bersih dari 3 kandidat, tapi checkpoint itu
sendiri cuma SATU sampel acak dari distribusi yang sama -- kebetulan di atas rata-rata.

**Kenapa ensemble, bukan retrain seed baru:** retrain sekali lagi cuma menghasilkan SATU
sampel baru lagi dari distribusi yang sama (std=0,0171 tidak hilang). Ensemble -- rata-rata
probabilitas dari beberapa model independen -- justru MENGURANGI variance prediksi akhir
("deep ensembles", Lakshminarayanan et al. 2017), bukan cuma menghindari kesan cherry-picking.
Trade-off: 5x biaya inferensi & kompleksitas deployment (5 checkpoint, bukan 1) -- wajar untuk
tugas sortir bean yang tidak latency-critical.

**Cakupan:**
1. Retrain `09_noise_robust` di 5 seed yang SAMA seperti `CBQD - HPO Seed Stability.ipynb`
   (42, 43, 44, 45, 46) -- kali ini checkpoint tiap seed DISIMPAN (`*_ensemble_seed{seed}.pt`),
   berbeda dari notebook itu yang sengaja tidak menyimpan checkpoint.
2. Verifikasi individual test-F1 tiap seed harus dekat dengan angka yang sudah tercatat di
   `metadata/hpo_seed_stability_details.csv` (sanity check -- retrain ini bukan eksperimen baru,
   cuma reproduksi + simpan checkpoint).
3. Bangun ensemble (rata-rata softmax 5 model), evaluasi macro-F1 & recall per kelas di test set
   yang sama, bandingkan ke mean/std/best-individual dari seed sweep.
4. Push 5 checkpoint ke R2/DVC sebagai artefak produksi resmi.


## Section 1 -- Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")


In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset + manifest dari R2 (dvc pull)

!pip install -q "dvc[s3]"
!dvc pull -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

CKPT_DIR = Path("models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)


## Section 2 -- Konfigurasi

In [ ]:
# Sub-Step 2.1
# Tujuan: DRY_RUN + SEEDS (identik CBQD - HPO Seed Stability.ipynb, supaya sebanding)

import random
import numpy as np
import torch

DRY_RUN = False  # <-- dry-run (v2) sudah diverifikasi bersih di Kaggle (COMPLETE, checkpoint tersimpan benar), full run.

if DRY_RUN:
    SEEDS = [42, 43]
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
else:
    SEEDS = [42, 43, 44, 45, 46]
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45
    EARLY_STOP_PATIENCE = 10

IMG_SIZE = 224
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEEDS[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | SEEDS={SEEDS}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "P100" in gpu_name:
        raise RuntimeError(
            f"GPU allocated is {gpu_name}, incompatible with the preinstalled PyTorch "
            "build (no Pascal/sm_60 kernels). Re-push the kernel with "
            "kernel-metadata.json machine_shape=NvidiaTeslaT4 to force a T4 allocation."
        )


In [ ]:
# Sub-Step 2.2
# Tujuan: Muat best hyperparameter 09_noise_robust (TETAP, identik seed-sweep sebelumnya)

import pandas as pd

hpo_summary = pd.read_csv("metadata/hpo_final_summary.csv").set_index("model")
_row = hpo_summary.loc["09_noise_robust"]
bp_noise_robust = {
    "lr_phase1": float(_row["param_lr_phase1"]), "lr_phase2": float(_row["param_lr_phase2"]),
    "weight_decay": float(_row["param_weight_decay"]), "batch_size": int(_row["param_batch_size"]),
    "scheduler_factor": float(_row["param_scheduler_factor"]), "scheduler_patience": int(_row["param_scheduler_patience"]),
    "label_smoothing": float(_row["param_label_smoothing"]), "mislabel_weight": float(_row["param_mislabel_weight"]),
    "mistake_threshold": float(_row["param_mistake_threshold"]),
}
print("[09_noise_robust] best params:", bp_noise_robust)


## Section 3 -- Data: Manifest, Dataset, Transform, DataLoader

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test/real_world DataFrame -- split identik
# seluruh notebook sebelumnya

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
real_world_df = manifest[manifest["split"] == "real_world"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}  real_world={len(real_world_df)}")


In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform, DataLoader test/real_world TETAP + make_loaders() per seed

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
EVAL_BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    def __init__(self, df, root_dir, transform, weights=None, label_fn=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        label_fn = label_fn if label_fn is not None else (lambda l: LABEL_TO_IDX[l])
        self.labels = [label_fn(l) for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


def make_loaders(fit_kwargs, val_kwargs, batch_size):
    fit_loader = DataLoader(BeanDataset(fit_df, PREP_DIR, train_transform, **fit_kwargs),
                             batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(BeanDataset(val_df, PREP_DIR, eval_transform, **val_kwargs),
                             batch_size=batch_size, shuffle=False, num_workers=2)
    return fit_loader, val_loader


test_loader = DataLoader(BeanDataset(test_df, PREP_DIR, eval_transform),
                          batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)
real_world_loader = DataLoader(
    BeanDataset(real_world_df, PREP_DIR, eval_transform, label_fn=lambda l: 0),
    batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2,
)
print("DataLoaders siap.")


## Section 4 -- Fungsi Utilitas Model

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate() -- untuk model tunggal maupun ensemble (lewat predict_proba eksternal)

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


@torch.no_grad()
def predict_proba_loader(model, loader):
    model.eval()
    all_proba = []
    for images, _, _ in loader:
        all_proba.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(all_proba, axis=0)


def metrics_from_proba(proba, labels):
    preds = proba.argmax(axis=1)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    report = classification_report(labels, preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report}


In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() + train_one_model_hpo() (identik notebook sebelumnya)

import copy


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def _train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()


def train_one_model_hpo(model, head_module, fit_loader, val_loader, device, model_name,
                         lr_phase1, lr_phase2, weight_decay, scheduler_factor, scheduler_patience,
                         epochs_phase2, criterion=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr_phase1, weight_decay=weight_decay
    )
    for epoch in range(EPOCHS_PHASE1):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience)
    patience_counter = 0
    for epoch in range(epochs_phase2):
        _train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["macro_f1"])
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1


In [ ]:
# Sub-Step 4.3
# Tujuan: build_model() -- efficientnet_b0

from torchvision import models as tv_models


def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


In [ ]:
# Sub-Step 4.4
# Tujuan: handcrafted_features()/build_feature_matrix() -- untuk mistake_score (fixed, identik seed-sweep)

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    else:
        area_frac = bbox_ratio = center_offset = np.nan

    edges = cv2.Canny(gray, 100, 200)
    feats = {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }
    return feats


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    rows = [handcrafted_features(RAW_DIR / p) for p in df["orig_path"]]
    X = pd.DataFrame(rows)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


## Section 5 -- Retrain 5 Seed, Checkpoint DISIMPAN Kali Ini

Hyperparameter &amp; seed identik `CBQD - HPO Seed Stability.ipynb` -- angka test-F1 tiap seed
DIHARAPKAN dekat dengan yang sudah tercatat di `metadata/hpo_seed_stability_details.csv`
(42=0,9304; 43=0,9568; 44=0,9435; 45=0,9697; 46=0,9697) sebagai sanity check. Bedanya cuma
satu: checkpoint disimpan ke `models/checkpoints/09_noise_robust_ensemble_seed{seed}.pt`.

In [ ]:
# Sub-Step 5.1
# Tujuan: Precompute mistake_score (TETAP, identik seed-sweep) untuk seluruh seed

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

X_fit_m9, y_fit_m9 = build_feature_matrix(fit_df)
_sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
_rf_noise = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
_proba_oof = cross_val_predict(_rf_noise, X_fit_m9, y_fit_m9, cv=_sgkf_noise,
                                groups=fit_df["cluster_id"].values, method="predict_proba")
_true_proba = _proba_oof[np.arange(len(y_fit_m9)), y_fit_m9]
_max_proba = _proba_oof.max(axis=1)
MISTAKE_SCORE = _max_proba - _true_proba

flagged_mask = MISTAKE_SCORE > bp_noise_robust["mistake_threshold"]
sample_weights_fit = np.where(flagged_mask, bp_noise_robust["mislabel_weight"], 1.0)
print(f"[09_noise_robust] kandidat mislabel: {flagged_mask.sum()} / {len(fit_df)} -- TETAP sepanjang seed sweep")


In [ ]:
# Sub-Step 5.2
# Tujuan: Retrain 5 seed, simpan checkpoint tiap seed, catat metrik individual

individual_results = []
ensemble_test_proba = []   # dikumpulkan langsung di sini -- hindari load ulang checkpoint di Section 6
ensemble_rw_proba = []

for seed in SEEDS:
    ckpt_path = CKPT_DIR / f"09_noise_robust_ensemble_seed{seed}.pt"
    set_all_seeds(seed)
    fit_loader, val_loader = make_loaders({"weights": sample_weights_fit.tolist()}, {}, bp_noise_robust["batch_size"])
    criterion = nn.CrossEntropyLoss(label_smoothing=bp_noise_robust["label_smoothing"], reduction="none")
    model, head = build_model("efficientnet_b0", 4)

    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model = model.to(device).eval()
        print(f"[seed={seed}] checkpoint dimuat dari {ckpt_path}")
    else:
        model, val_f1 = train_one_model_hpo(
            model, head, fit_loader, val_loader, device, f"09_noise_robust_ensemble_seed{seed}",
            lr_phase1=bp_noise_robust["lr_phase1"], lr_phase2=bp_noise_robust["lr_phase2"],
            weight_decay=bp_noise_robust["weight_decay"], scheduler_factor=bp_noise_robust["scheduler_factor"],
            scheduler_patience=bp_noise_robust["scheduler_patience"], epochs_phase2=EPOCHS_PHASE2, criterion=criterion,
        )
        torch.save(model.state_dict(), ckpt_path)
        print(f"[seed={seed}] selesai dilatih & disimpan ke {ckpt_path}, val_f1={val_f1:.4f}")

    test_proba = predict_proba_loader(model, test_loader)
    rw_proba = predict_proba_loader(model, real_world_loader)
    test_metrics = metrics_from_proba(test_proba, test_df["label"].map(LABEL_TO_IDX).values)
    individual_results.append({"seed": seed, "test_macro_f1": test_metrics["macro_f1"],
                                "test_accuracy": test_metrics["accuracy"]})
    ensemble_test_proba.append(test_proba)
    ensemble_rw_proba.append(rw_proba)
    print(f"[seed={seed}] test_macro_f1={test_metrics['macro_f1']:.4f}")
    del model
    torch.cuda.empty_cache()

individual_df = pd.DataFrame(individual_results)
print()
print(individual_df.round(4).to_string(index=False))


## Section 6 -- Bangun & Evaluasi Ensemble

In [ ]:
# Sub-Step 6.1
# Tujuan: Rata-ratakan probabilitas softmax 5 model -- evaluasi test & real_world

y_test_true = test_df["label"].map(LABEL_TO_IDX).values

ensemble_test_proba_mean = np.mean(ensemble_test_proba, axis=0)
ensemble_rw_proba_mean = np.mean(ensemble_rw_proba, axis=0)

ensemble_metrics = metrics_from_proba(ensemble_test_proba_mean, y_test_true)
print(f"[ENSEMBLE 5-seed] test_macro_f1={ensemble_metrics['macro_f1']:.4f}  "
      f"test_accuracy={ensemble_metrics['accuracy']:.4f}")
for cls in CLASS_NAMES:
    print(f"  recall {cls}: {ensemble_metrics['report'][cls]['recall']:.4f}")

ensemble_rw_conf = ensemble_rw_proba_mean.max(axis=1).mean()
print(f"[ENSEMBLE 5-seed] real_world confidence mean: {ensemble_rw_conf:.4f}")


In [ ]:
# Sub-Step 6.2
# Tujuan: Bandingkan ensemble ke individual seed & ke referensi seed-sweep sebelumnya

seed_stability_ref = pd.read_csv("metadata/hpo_seed_stability_summary.csv").set_index("model").loc["09_noise_robust"]

comparison_rows = [{"variant": f"seed_{r['seed']}", "test_macro_f1": r["test_macro_f1"]} for r in individual_results]
comparison_rows.append({"variant": "ENSEMBLE (5-seed mean-proba)", "test_macro_f1": ensemble_metrics["macro_f1"]})
comparison_rows.append({"variant": "individual_mean (seed-sweep lama)", "test_macro_f1": seed_stability_ref["test_macro_f1_mean"]})
comparison_rows.append({"variant": "individual_best (max seed di run ini)", "test_macro_f1": individual_df["test_macro_f1"].max()})
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.round(4).to_string(index=False))

ensemble_vs_mean = ensemble_metrics["macro_f1"] - seed_stability_ref["test_macro_f1_mean"]
ensemble_vs_best = ensemble_metrics["macro_f1"] - individual_df["test_macro_f1"].max()
print()
print(f"Ensemble vs mean individual (seed-sweep lama): {ensemble_vs_mean:+.4f}")
print(f"Ensemble vs best individual (run ini): {ensemble_vs_best:+.4f}")


## Section 7 -- Push Checkpoint Ensemble ke R2/DVC

In [ ]:
# Sub-Step 7.1
# Tujuan: Push 5 checkpoint ensemble ke R2/DVC

_new_ckpts = [CKPT_DIR / f"09_noise_robust_ensemble_seed{seed}.pt" for seed in SEEDS]
_all_exist_before = all(p.exists() for p in _new_ckpts)
os.system("dvc add models/checkpoints")
os.system("dvc push")
dvc_file = Path("models/checkpoints.dvc")
if dvc_file.exists():
    print("--- Isi models/checkpoints.dvc (salin ke repo lokal, lalu commit) ---")
    print(dvc_file.read_text())


## Section 8 -- Konsolidasi & Kesimpulan

In [ ]:
# Sub-Step 8.1
# Tujuan: Simpan ringkasan akhir + interpretasi

Path("metadata").mkdir(exist_ok=True)

individual_df.to_csv("metadata/09_noise_robust_ensemble_individual.csv", index=False)

summary_row = {
    "ensemble_test_macro_f1": ensemble_metrics["macro_f1"],
    "ensemble_test_accuracy": ensemble_metrics["accuracy"],
    "ensemble_real_world_confidence": ensemble_rw_conf,
    "individual_mean_seed_sweep": seed_stability_ref["test_macro_f1_mean"],
    "individual_std_seed_sweep": seed_stability_ref["test_macro_f1_std"],
    "individual_best_this_run": individual_df["test_macro_f1"].max(),
    "individual_worst_this_run": individual_df["test_macro_f1"].min(),
    "ensemble_vs_mean_delta": ensemble_vs_mean,
    "ensemble_vs_best_delta": ensemble_vs_best,
    "n_seeds": len(SEEDS),
}
for cls in CLASS_NAMES:
    summary_row[f"ensemble_recall_{cls}"] = ensemble_metrics["report"][cls]["recall"]

summary_df = pd.DataFrame([summary_row])
summary_df.to_csv("metadata/09_noise_robust_ensemble_summary.csv", index=False)

status_text = "BELUM final (seed/epoch kecil)" if DRY_RUN else "hasil run penuh"
print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {status_text}")
print()
pd.set_option("display.max_columns", None, "display.width", 250)
print(summary_df.round(4).T.to_string())


In [ ]:
# Sub-Step 8.2
# Tujuan: Interpretasi -- apakah ensemble ini layak jadi artefak produksi final

print(f"""
=== Kesimpulan: 09_noise_robust -- Ensemble 5-Seed sebagai Artefak Produksi ===

Ensemble test macro-F1: {ensemble_metrics['macro_f1']:.4f}
  vs mean individual (seed-sweep, n=5 lama): {seed_stability_ref['test_macro_f1_mean']:.4f} (delta {ensemble_vs_mean:+.4f})
  vs best individual (run ini, n={len(SEEDS)}): {individual_df['test_macro_f1'].max():.4f} (delta {ensemble_vs_best:+.4f})
  vs std individual (seed-sweep lama): {seed_stability_ref['test_macro_f1_std']:.4f}

Cara membaca:
- Ensemble yang mengalahkan atau setara mean individual, TAPI dengan proses yang tidak
  bergantung pada satu seed tertentu, adalah bukti kuat bahwa merata-ratakan prediksi
  berhasil "membatalkan" sebagian noise training yang sudah dikonfirmasi ada
  (std={seed_stability_ref['test_macro_f1_std']:.4f} dari seed-sweep).
- Ensemble TIDAK HARUS mengalahkan best-individual (0,9697/0,9826 di seed-sweep) untuk
  dianggap berhasil -- best-individual adalah hasil UNTUNG dari satu seed, bukan target yang
  realistis untuk dikejar ulang. Target ensemble yang wajar adalah: dekat atau di atas MEAN,
  dengan risiko downside (kalau salah satu model "kalah") jauh lebih kecil dibanding
  bergantung pada satu checkpoint saja.
- Trade-off yang diterima dengan memilih ensemble: 5x biaya komputasi inferensi
  dibanding 1 model saja, dan deployment perlu memuat & menjalankan 5 checkpoint
  (models/checkpoints/09_noise_robust_ensemble_seed{{42,43,44,45,46}}.pt), lalu
  merata-ratakan softmax-nya sebelum argmax.

REKOMENDASI FINAL: gunakan ensemble 5-seed ini (bukan checkpoint tunggal manapun, termasuk
09_noise_robust_tuned.pt yang sudah lolos XAI penuh) sebagai artefak produksi -- variance
run-to-run yang sudah terbukti nyata (CBQD - HPO Seed Stability.ipynb) membuat SATU checkpoint
manapun secara inheren tidak bisa mewakili performa "sebenarnya" model ini; ensemble adalah
cara paling defensible untuk tidak bergantung pada satu tebakan acak.
""")
